In [100]:
from langgraph.graph import StateGraph,START,END
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict , Literal , Annotated
from dotenv import load_dotenv
from langchain_core.messages import SystemMessage, HumanMessage

In [101]:
load_dotenv()

True

In [102]:
generator_llm=ChatGoogleGenerativeAI(model="gemini-2.0-flash")
evaluator_llm=ChatGoogleGenerativeAI(model="gemini-2.5-flash")
optimizer_llm=ChatGoogleGenerativeAI(model="gemini-2.0-flash")

In [103]:
from pydantic import BaseModel, Field
class TweetEvaluator(BaseModel):
   evaluation: Literal["approved","need_improvement"] = Field(..., description="final evaluation of the tweet")
   feedback: str = Field(..., description="constructive feedback for the tweet")

In [104]:
structure_llm_evaluator=evaluator_llm.with_structured_output(TweetEvaluator)

In [105]:
class PostState(TypedDict):
    topic:str
    tweet:str
    evaluation : Literal['approved','needs_improvement']
    feedback: str
    iteration: int
    max_iteration: int

In [106]:
def generate(State: PostState):
    #prompt
    messages = [
    SystemMessage(
        content="You are a funny and clever Twitter/X influencer."
    ),
    HumanMessage(
        content=f"""
Write a short, original, and hilarious tweet on the topic: "{State['topic']}".

Rules:
- Do NOT use question-answer format.
- Max 280 characters.
- Use observational humor, irony, sarcasm, or cultural references.
- Think in meme logic, punchlines, or relatable takes.
- Use simple, day to day english.

"""
    )
]
    response = generator_llm.invoke(messages).content
    return {"tweet": response}


In [107]:
def evaluate(State: PostState):
    messages = [
    SystemMessage(
        content="You are a ruthless, no-laugh-given Twitter critic. You evaluate tweets based on humor, originality, virality, and tweet format."
    ),
    HumanMessage(
        content=f"""
Evaluate the following tweet:

Tweet: "{State['tweet']}"

Use the criteria below to evaluate the tweet:

1. Originality - Is this fresh, or have you seen it a hundred times before?
2. Humor - Did it genuinely make you smile, laugh, or chuckle?
3. Punchiness - Is it short, sharp, and scroll-stopping?
4. Virality Potential - Would people retweet or share it?
5. Format - Is it a well-formed tweet (not a setup-punchline joke, not a Q&A joke, and under 280 characters)?

Auto-reject if:
- It's written in question-answer format (e.g., "Why did..." or "What happens when...")
- It exceeds 280 characters
- It reads like a traditional setup-punchline joke
- Don't end with generic, throwaway, or deflating lines that weaken the humor (e.g., "Masterpieces of the auntie-uncle universe" or vague summaries)

### Respond ONLY in structured format:
- evaluation: "approved" or "needs_improvement"
- feedback: One paragraph explaining the strengths and weaknesses
"""
    )
]
    response = structure_llm_evaluator.invoke(messages)
    return {"evaluation": response.evaluation, "feedback": response.feedback}

    

In [108]:
def optimise(State: PostState):
    messages = [
    SystemMessage(
        content="You punch up tweets for virality and humor based on given feedback."
    ),
    HumanMessage(
        content=f"""
Improve the tweet based on this feedback:
"{State['feedback']}"

Topic: "{State['topic']}"

Original Tweet:
{State['tweet']}

Re-write it as a short, viral-worthy tweet. Avoid Q&A style and stay under 280 characters.
"""
    )
]
    response = optimizer_llm.invoke(messages)
    iteration = State['iteration'] + 1
    return {"tweet": response.content, "iteration": iteration}

In [109]:
def check_improvement(State: PostState):
    if State['evaluation'] == 'approved' or State['iteration'] >= State['max_iteration']:
        return 'approved'
    
    else:
        return 'needs_improvement'

In [110]:
approved = 'approved'
needs_improvement = 'needs_improvement'

In [111]:
graph=StateGraph(PostState)

graph.add_node("generate",generate)
graph.add_node("evaluate",evaluate)
graph.add_node("optimise",optimise)

graph.add_edge(START,"generate")
graph.add_edge("generate","evaluate")

graph.add_conditional_edges("evaluate",check_improvement,{approved:END, needs_improvement:"optimise"})

graph.add_edge("optimise","evaluate")

workflow=graph.compile()

In [ ]:
initial_state = {
    "topic": "ai",
    "iteration":1,
    "max_iteration":3,
}
workflow.invoke(initial_state)